In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers

In [ ]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
os.environ["GOOGLE_API_KEY"] = "xxxxxxxxxxxx"

In [ ]:
print(f"{len(documents)} documentos cargados.\n")

for i, doc in enumerate(documents[:10]):
    print("=" * 80)
    print(f"Documento {i+1}")
    print("SOURCE:")
    print(doc.metadata.get("source"))
    print("\nPrimeros 300 caracteres:")
    print(doc.page_content[:300])
    print()

248 documentos cargados.

Documento 1
SOURCE:
data/raw/OECD_AI_Principles_ES.pdf.pdf

Primeros 300 caracteres:
OECD  AI  Principles  
Metadatos  del  Documento  ●  Fuente  oficial:  https://oecd.ai/en/ai-principles ●  Fecha  de  consulta:  27/06/2026  
Resumen  de  los  Principios  de  la  OCDE  
Los  Principios  de  la  OCDE  sobre  IA  promueven  el  uso  de  una  inteligencia  artificial  
innovadora
 
y


Documento 2
SOURCE:
data/raw/OECD_AI_Principles_ES.pdf.pdf

Primeros 300 caracteres:
1.5  -  Responsabilidad  (Accountability)  Las  organizaciones  e  individuos  que  desarrollan  u  operan  sistemas  de  IA  deben  rendir  
cuentas
 
de
 
su
 
funcionamiento,
 
garantizando
 
la
 
trazabilidad
 
de
 
los
 
procesos
 
y
 
aplicando
 
una
 
gestión
 
sistemática
 
de
 
riesgos.
 
R

Documento 3
SOURCE:
data/raw/UNESCO_Ethics_of_AI_Es.pdf.pdf

Primeros 300 caracteres:
Recomendación sobre   
la ética de 
la inteligencia 
artificial
Adoptada el 23 de noviembre de 2021

Documento 4
S

In [ ]:
print("Dividiendo documentos...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"{len(chunks)} chunks creados.")

Dividiendo documentos...
813 chunks creados.


In [ ]:
print("Creando embeddings locales...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Creando embeddings locales...


/tmp/ipykernel_32174/310929616.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
print("Construyendo índice FAISS...")
print("Cantidad de chunks:", len(chunks))

print("Primer chunk:")
print(chunks[0].page_content[:200])

texto_prueba = [chunks[0].page_content]

emb = embeddings.embed_documents(texto_prueba)

print("Embeddings devueltos:")
print(emb)

print("Cantidad:", len(emb))
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

Construyendo índice FAISS...
Cantidad de chunks: 813
Primer chunk:
OECD  AI  Principles  
Metadatos  del  Documento  ●  Fuente  oficial:  https://oecd.ai/en/ai-principles ●  Fecha  de  consulta:  27/06/2026  
Resumen  de  los  Principios  de  la  OCDE  
Los  Principi
Embeddings devueltos:
[[-0.04106341302394867, 0.04126816242933273, -0.020508164539933205, -0.08000461012125015, 0.028241103515028954, 0.0017295057186856866, 0.001901166862808168, 0.08575005829334259, -0.008774392306804657, 0.0738719254732132, 0.07762396335601807, 0.028496775776147842, -0.015525306575000286, -0.02303249202668667, 0.017708033323287964, 0.03788751736283302, -0.03585989028215408, 0.02037646621465683, -0.0660717710852623, 0.05673994868993759, 0.11687782406806946, 0.0056564221158623695, 0.009994455613195896, 0.03539924696087837, -0.1318657547235489, -0.006719657685607672, -0.016365347430109978, -0.07709550112485886, -0.04214352369308472, 0.018842456862330437, 0.049198001623153687, 0.012725059874355793, 0.1080778

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [ ]:
def manual_rag(query: str):

    # Recuperar documentos
    docs = retriever.invoke(query)

    print("\nDOCUMENTOS RECUPERADOS\n")

    for i, doc in enumerate(docs):
        print(f"\n--- Documento {i+1} ---")
        print("SOURCE:", doc.metadata.get("source"))
        print("PAGE:", doc.metadata.get("page"))
        print(doc.page_content[:500])

    # Construir contexto
    contexto = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Prompt
    prompt = f"""
Sos un asistente que responde únicamente utilizando el CONTEXTO proporcionado.

Si la respuesta está en el contexto:
- Respondé en español.
- Citá los conceptos relevantes.
- No inventes información.

Si la respuesta NO está en el contexto respondé exactamente:

"La información no está disponible en los documentos consultados."

====================
CONTEXTO
====================

{contexto}

====================
PREGUNTA
====================

{query}

====================
RESPUESTA
====================
"""

    respuesta = llm.invoke(prompt)

    return respuesta.content

In [ ]:
print("\n=========================================")
print("Asistente listo.")
print("Escribí 'salir' para terminar.")
print("=========================================\n")

while True:

    pregunta = input("Pregunta: ")

    if pregunta.lower() == "salir":
        print("Hasta luego.")
        break

    respuesta = manual_rag(pregunta)

    print("\nRespuesta:\n")
    print(respuesta)
    print("\n" + "=" * 80 + "\n")


Asistente listo.
Escribí 'salir' para terminar.


DOCUMENTOS RECUPERADOS


--- Documento 1 ---
SOURCE: data/raw/OECD_AI_Principles_ES.pdf.pdf
PAGE: 0
OECD  AI  Principles  
Metadatos  del  Documento  ●  Fuente  oficial:  https://oecd.ai/en/ai-principles ●  Fecha  de  consulta:  27/06/2026  
Resumen  de  los  Principios  de  la  OCDE  
Los  Principios  de  la  OCDE  sobre  IA  promueven  el  uso  de  una  inteligencia  artificial  
innovadora
 
y
 
confiable
 
que
 
respete
 
los
 
derechos
 
humanos
 
y
 
los
 
valores
 
democráticos.
 
Adoptados
 
originalmente
 
en
 
mayo
 
de
 
2019
 
y
 
actualizados
 
en
 
mayo
 
de
 
2024,
 
establecen


--- Documento 2 ---
SOURCE: data/raw/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAGE: 39
género, con miras a mejorar los procesos de aprendizaje 
y fortalecer los nexos entre las conclusiones, la adopción 
de decisiones, la transparencia y la rendición de cuentas 
sobre los resultados.

--- Documento 3 ---
SOURCE: data/raw/UNESCO_Ethics_of_AI_Es.pdf.pdf
PAG